In [3]:

import os
import csv
import math
import random
import urllib.request
from collections import Counter


In [4]:
# Download AG News dataset

def download_ag_news():

    # URLs for AG News dataset
    train_url = "https://raw.githubusercontent.com/mhjabreel/CharCnn_Keras/master/data/ag_news_csv/train.csv"
    test_url = "https://raw.githubusercontent.com/mhjabreel/CharCnn_Keras/master/data/ag_news_csv/test.csv"

    # Create data directory if not exists
    os.makedirs("data", exist_ok=True)

    print("Downloading AG News training data")

    try:
        urllib.request.urlretrieve(train_url, "data/ag_news_train.csv")
        print("  Downloaded: data/ag_news_train.csv")
    except Exception as e:
        print(f"  Error downloading train data: {e}")
        return None, None

    print("Downloading AG News test data")

    try:
        urllib.request.urlretrieve(test_url, "data/ag_news_test.csv")
        print("  Downloaded: data/ag_news_test.csv")
    except Exception as e:
        print(f"  Error downloading test data: {e}")
        return None, None

    return "data/ag_news_train.csv", "data/ag_news_test.csv"


In [5]:

# Load dataset and filter categories

def load_and_filter_data(filepath, categories={1: 'politics', 2: 'sports'}):

    data = []

    # Open CSV file
    with open(filepath, 'r', encoding='utf-8') as f:
        reader = csv.reader(f)

        for row in reader:

            # Ensure row structure is valid
            if len(row) >= 3:

                class_idx = int(row[0])
                title = row[1]
                description = row[2]

                # Keep only selected categories
                if class_idx in categories:

                    # Combine title and description
                    text = f"{title} {description}"
                    label = categories[class_idx]

                    data.append((text, label))

    return data


In [6]:
# Create training and testing files

def create_dataset_files(train_data, test_data, samples_per_class=500):

    # Ensure data directory exists
    os.makedirs("data", exist_ok=True)

    # Separate data by class
    train_sports = [t for t, l in train_data if l == 'sports']
    train_politics = [t for t, l in train_data if l == 'politics']

    test_sports = [t for t, l in test_data if l == 'sports']
    test_politics = [t for t, l in test_data if l == 'politics']

    # Shuffle datasets for randomness
    random.seed(42)
    random.shuffle(train_sports)
    random.shuffle(train_politics)
    random.shuffle(test_sports)
    random.shuffle(test_politics)

    # Limit dataset size
    train_sports = train_sports[:samples_per_class]
    train_politics = train_politics[:samples_per_class]

    test_sports = test_sports[:samples_per_class // 5]
    test_politics = test_politics[:samples_per_class // 5]

    # Save sports training data
    with open("data/sports_train.txt", 'w', encoding='utf-8') as f:
        for text in train_sports:
            f.write(text.replace('\n', ' ') + '\n')

    # Save politics training data
    with open("data/politics_train.txt", 'w', encoding='utf-8') as f:
        for text in train_politics:
            f.write(text.replace('\n', ' ') + '\n')

    # Save sports test data
    with open("data/sports_test.txt", 'w', encoding='utf-8') as f:
        for text in test_sports:
            f.write(text.replace('\n', ' ') + '\n')

    # Save politics test data
    with open("data/politics_test.txt", 'w', encoding='utf-8') as f:
        for text in test_politics:
            f.write(text.replace('\n', ' ') + '\n')

    print("\nDataset created:")
    print(f"  Training: {len(train_sports)} sports, {len(train_politics)} politics")
    print(f"  Testing:  {len(test_sports)} sports, {len(test_politics)} politics")

    return {
        'train_sports': train_sports,
        'train_politics': train_politics,
        'test_sports': test_sports,
        'test_politics': test_politics
    }


In [7]:
# Basic dataset analysis

def analyze_dataset(dataset):

    print("\n" + "=" * 60)
    print("DATASET ANALYSIS")
    print("=" * 60)

    # Combine train + test data
    all_sports = dataset['train_sports'] + dataset['test_sports']
    all_politics = dataset['train_politics'] + dataset['test_politics']

    # Compute statistics
    def get_stats(texts):
        word_counts = [len(text.split()) for text in texts]
        avg_words = sum(word_counts) / len(word_counts)
        return avg_words, min(word_counts), max(word_counts)

    sports_stats = get_stats(all_sports)
    politics_stats = get_stats(all_politics)

    print(f"\nSports articles ({len(all_sports)} total):")
    print(f"  Avg words: {sports_stats[0]:.1f}")

    print(f"\nPolitics articles ({len(all_politics)} total):")
    print(f"  Avg words: {politics_stats[0]:.1f}")


In [8]:
# Fallback sample data creation

def create_sample_data():

    os.makedirs("data", exist_ok=True)

    sports_samples = [
        "Lakers defeat Celtics in overtime thriller",
        "Manchester United signs new striker",
        "World Cup final draws record audience"
    ]

    politics_samples = [
        "Senate passes new infrastructure bill",
        "President meets foreign leaders",
        "Congress debates new legislation"
    ]

    with open("data/sports_train.txt", 'w') as f:
        f.write('\n'.join(sports_samples))

    with open("data/politics_train.txt", 'w') as f:
        f.write('\n'.join(politics_samples))

    print("Sample data created")


In [9]:
# Main execution pipeline

def main():

    print("=" * 60)
    print("STEP 1: DATA COLLECTION - AG News Dataset")
    print("=" * 60)

    # Download dataset
    train_path, test_path = download_ag_news()

    # Handle download failure
    if not train_path:
        print("Download failed. Using sample data.")
        create_sample_data()
        return

    print("\nLoading and filtering data...")

    # Load filtered data
    train_data = load_and_filter_data(train_path)
    test_data = load_and_filter_data(test_path)

    print(f"  Train: {len(train_data)} articles")
    print(f"  Test: {len(test_data)} articles")

    # Create dataset files
    dataset = create_dataset_files(train_data, test_data)

    # Analyze dataset
    analyze_dataset(dataset)

    print("\nStep 1 complete!")


In [10]:
if __name__ == "__main__":
    main()

STEP 1: DATA COLLECTION - AG News Dataset
  Downloaded: data/ag_news_train.csv
  Downloaded: data/ag_news_test.csv

Loading and filtering data...
  Train: 60000 articles
  Test: 3800 articles

Dataset created:
  Training: 500 sports, 500 politics
  Testing:  100 sports, 100 politics

DATASET ANALYSIS

Sports articles (600 total):
  Avg words: 37.6

Politics articles (600 total):
  Avg words: 38.7

Step 1 complete!


In [1]:
if __name__ == "__main__":
    main()

  STEP 1: DATA COLLECTION - AG News Dataset

  Downloaded: data/ag_news_train.csv
  Downloaded: data/ag_news_test.csv

Loading and filtering data (Sports & Politics only)...
  Train: 60000 articles
  Test: 3800 articles

Dataset created:
  Training: 500 sports, 500 politics
  Testing:  100 sports, 100 politics

Files saved in 'data/' folder:
  - sports_train.txt
  - politics_train.txt
  - sports_test.txt
  - politics_test.txt

DATASET ANALYSIS

Sports articles (600 total):
  Avg words per article: 37.6
  Min words: 8, Max words: 78

Politics articles (600 total):
  Avg words per article: 38.7
  Min words: 11, Max words: 87

Most common words in SPORTS:
  39s: 195
  new: 99
  first: 84
  game: 82
  win: 72
  team: 72
  league: 63
  victory: 60
  one: 59
  out: 57
  against: 57
  world: 56
  two: 55
  season: 55
  night: 54

Most common words in POLITICS:
  said: 152
  39s: 141
  iraq: 132
  reuters: 100
  new: 80
  minister: 75
  president: 74
  killed: 74
  two: 68
  afp: 68
  prime: 5

In [ ]:
# Basic text cleaning
def preprocess(text):
    text = text.lower()

    # Keep only letters and digits
    cleaned = ''.join(c if c.isalnum() or c.isspace() else ' ' for c in text)

    return cleaned.split()


# Generate n-grams
def get_ngrams(tokens, n):
    ngrams = []

    for i in range(len(tokens) - n + 1):
        ngram = '_'.join(tokens[i:i+n])
        ngrams.append(ngram)

    return ngrams


In [ ]:
class BagOfWords:

    def __init__(self, max_features=5000, ngram_range=(1, 1)):
        self.max_features = max_features
        self.ngram_range = ngram_range
        self.vocabulary = {}

    # Collect all n-grams
    def _get_all_ngrams(self, tokens):
        all_ngrams = []

        for n in range(self.ngram_range[0], self.ngram_range[1] + 1):
            if n == 1:
                all_ngrams.extend(tokens)
            else:
                all_ngrams.extend(get_ngrams(tokens, n))

        return all_ngrams

    # Build vocabulary
    def fit(self, documents):
        word_counts = Counter()

        for doc in documents:
            tokens = preprocess(doc)
            ngrams = self._get_all_ngrams(tokens)

            # Document frequency
            word_counts.update(set(ngrams))

        most_common = word_counts.most_common(self.max_features)
        self.vocabulary = {word: idx for idx, (word, _) in enumerate(most_common)}

        return self

    # Convert documents into vectors
    def transform(self, documents):
        vectors = []

        for doc in documents:
            tokens = preprocess(doc)
            ngrams = self._get_all_ngrams(tokens)
            ngram_counts = Counter(ngrams)

            vector = [0] * len(self.vocabulary)

            for word, idx in self.vocabulary.items():
                if word in ngram_counts:
                    vector[idx] = ngram_counts[word]

            vectors.append(vector)

        return vectors

    def fit_transform(self, documents):
        self.fit(documents)
        return self.transform(documents)


In [ ]:
class TfIdfVectorizer:

    def __init__(self, max_features=5000, ngram_range=(1, 1)):
        self.max_features = max_features
        self.ngram_range = ngram_range
        self.vocabulary = {}
        self.idf = {}

    def _get_all_ngrams(self, tokens):
        all_ngrams = []

        for n in range(self.ngram_range[0], self.ngram_range[1] + 1):
            if n == 1:
                all_ngrams.extend(tokens)
            else:
                all_ngrams.extend(get_ngrams(tokens, n))

        return all_ngrams

    # Learn vocabulary + IDF
    def fit(self, documents):
        doc_freq = Counter()
        n_docs = len(documents)

        for doc in documents:
            tokens = preprocess(doc)
            ngrams = self._get_all_ngrams(tokens)

            doc_freq.update(set(ngrams))

        most_common = doc_freq.most_common(self.max_features)
        self.vocabulary = {word: idx for idx, (word, _) in enumerate(most_common)}

        # Compute smoothed IDF
        for word in self.vocabulary:
            df = doc_freq[word]
            self.idf[word] = math.log(n_docs / (df + 1)) + 1

        return self

    # Transform documents
    def transform(self, documents):
        vectors = []

        for doc in documents:
            tokens = preprocess(doc)
            ngrams = self._get_all_ngrams(tokens)
            ngram_counts = Counter(ngrams)

            total = len(ngrams) if ngrams else 1
            vector = [0.0] * len(self.vocabulary)

            for word, idx in self.vocabulary.items():
                if word in ngram_counts:
                    tf = ngram_counts[word] / total
                    vector[idx] = tf * self.idf[word]

            # L2 normalization
            norm = math.sqrt(sum(v*v for v in vector)) or 1
            vector = [v / norm for v in vector]

            vectors.append(vector)

        return vectors

    def fit_transform(self, documents):
        self.fit(documents)
        return self.transform(documents)


In [ ]:
class NaiveBayesClassifier:

    def __init__(self, alpha=1.0):
        self.alpha = alpha
        self.class_priors = {}
        self.feature_probs = {}
        self.classes = []

    def fit(self, X, y):
        self.classes = list(set(y))
        n_samples = len(y)
        n_features = len(X[0])

        for c in self.classes:
            class_samples = [X[i] for i in range(n_samples) if y[i] == c]

            self.class_priors[c] = len(class_samples) / n_samples

            feature_sums = [0] * n_features
            for sample in class_samples:
                for i in range(n_features):
                    feature_sums[i] += sample[i]

            total = sum(feature_sums)

            self.feature_probs[c] = [
                (feature_sums[i] + self.alpha) / (total + self.alpha * n_features)
                for i in range(n_features)
            ]

        return self

    def predict(self, X):
        predictions = []

        for sample in X:
            best_class = None
            best_score = float('-inf')

            for c in self.classes:
                score = math.log(self.class_priors[c])

                for i, val in enumerate(sample):
                    if val > 0:
                        score += val * math.log(self.feature_probs[c][i])

                if score > best_score:
                    best_score = score
                    best_class = c

            predictions.append(best_class)

        return predictions


In [ ]:
class LogisticRegression:

    def __init__(self, lr=0.1, epochs=100, regularization=0.01):
        self.lr = lr
        self.epochs = epochs
        self.reg = regularization
        self.weights = None
        self.bias = 0.0

    def _sigmoid(self, z):
        z = max(-500, min(500, z))
        return 1 / (1 + math.exp(-z))

    def fit(self, X, y):
        n_samples = len(X)
        n_features = len(X[0])

        self.weights = [0.0] * n_features
        self.bias = 0.0

        for epoch in range(self.epochs):
            dw = [0.0] * n_features
            db = 0.0

            for i in range(n_samples):
                z = sum(X[i][j] * self.weights[j] for j in range(n_features)) + self.bias
                pred = self._sigmoid(z)

                error = pred - y[i]

                for j in range(n_features):
                    dw[j] += error * X[i][j]

                db += error

            for j in range(n_features):
                self.weights[j] -= self.lr * (dw[j]/n_samples + self.reg * self.weights[j])

            self.bias -= self.lr * db / n_samples

        return self

    def predict(self, X):
        predictions = []
        n_features = len(X[0])

        for sample in X:
            z = sum(sample[j] * self.weights[j] for j in range(n_features)) + self.bias
            prob = self._sigmoid(z)

            predictions.append(1 if prob >= 0.5 else 0)

        return predictions


In [ ]:
class SVM:

    def __init__(self, lr=0.001, epochs=100, C=1.0):
        self.lr = lr
        self.epochs = epochs
        self.C = C
        self.weights = None
        self.bias = 0.0

    def fit(self, X, y):
        n_samples = len(X)
        n_features = len(X[0])

        y_svm = [1 if label == 1 else -1 for label in y]

        self.weights = [0.0] * n_features
        self.bias = 0.0

        for epoch in range(self.epochs):
            for i in range(n_samples):
                z = sum(X[i][j] * self.weights[j] for j in range(n_features)) + self.bias

                if y_svm[i] * z < 1:
                    for j in range(n_features):
                        self.weights[j] -= self.lr * (
                            2 * (1/self.epochs) * self.weights[j] - self.C * y_svm[i] * X[i][j]
                        )

                    self.bias -= self.lr * (-self.C * y_svm[i])

                else:
                    for j in range(n_features):
                        self.weights[j] -= self.lr * (2 * (1/self.epochs) * self.weights[j])

        return self

    def predict(self, X):
        predictions = []
        n_features = len(X[0])

        for sample in X:
            z = sum(sample[j] * self.weights[j] for j in range(n_features)) + self.bias
            predictions.append(1 if z >= 0 else 0)

        return predictions


In [ ]:
def compute_metrics(y_true, y_pred):
    tp = sum(1 for t, p in zip(y_true, y_pred) if t == 1 and p == 1)
    tn = sum(1 for t, p in zip(y_true, y_pred) if t == 0 and p == 0)
    fp = sum(1 for t, p in zip(y_true, y_pred) if t == 0 and p == 1)
    fn = sum(1 for t, p in zip(y_true, y_pred) if t == 1 and p == 0)

    accuracy = (tp + tn) / len(y_true)
    precision = tp / (tp + fp) if (tp + fp) else 0
    recall = tp / (tp + fn) if (tp + fn) else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0

    return accuracy, precision, recall, f1


In [2]:



def print_results(name, metrics):
    """Pretty print results."""
    print(f"\n{name}")
    print("-" * 40)
    print(f"  Accuracy:  {metrics['accuracy']:.4f}")
    print(f"  Precision: {metrics['precision']:.4f}")
    print(f"  Recall:    {metrics['recall']:.4f}")
    print(f"  F1-Score:  {metrics['f1']:.4f}")
    cm = metrics['confusion_matrix']
    print(f"  Confusion Matrix:")
    print(f"    TP={cm['tp']}, TN={cm['tn']}, FP={cm['fp']}, FN={cm['fn']}")

# MAIN EXPERIMENT

def run_experiment():
    print("=" * 70)
    print("   SPORTS vs POLITICS CLASSIFIER")
    print("   Comparing: Naive Bayes, Logistic Regression, SVM")
    print("   Features: Bag of Words, TF-IDF, N-grams")
    print("=" * 70)

    # Load data
    print("\n[1] Loading AG News dataset...")
    train_path, test_path = download_ag_news()

    X_train, y_train = load_ag_news(train_path, max_per_class=800)
    X_test, y_test = load_ag_news(test_path, max_per_class=200)

    print(f"    Training samples: {len(X_train)} (Politics: {y_train.count(0)}, Sports: {y_train.count(1)})")
    print(f"    Test samples: {len(X_test)} (Politics: {y_test.count(0)}, Sports: {y_test.count(1)})")

    # Store all results for comparison
    results = []

    #  EXPERIMENT 1: Bag of Words
    print("\n[2] Feature Extraction: Bag of Words (Unigrams)")
    bow = BagOfWords(max_features=3000, ngram_range=(1, 1))
    X_train_bow = bow.fit_transform(X_train)
    X_test_bow = bow.transform(X_test)
    print(f"    Vocabulary size: {len(bow.vocabulary)}")

    # Naive Bayes + BoW
    print("\n[3] Training classifiers on BoW features...")

    nb = NaiveBayesClassifier(alpha=1.0)
    nb.fit(X_train_bow, y_train)
    y_pred = nb.predict(X_test_bow)
    metrics = compute_metrics(y_test, y_pred)
    print_results("Naive Bayes + Bag of Words", metrics)
    results.append(("Naive Bayes", "BoW", metrics))

    # Logistic Regression + BoW
    lr = LogisticRegression(lr=0.5, epochs=100)
    lr.fit(X_train_bow, y_train)
    y_pred = lr.predict(X_test_bow)
    metrics = compute_metrics(y_test, y_pred)
    print_results("Logistic Regression + Bag of Words", metrics)
    results.append(("Logistic Regression", "BoW", metrics))

    # SVM + BoW
    svm = SVM(lr=0.001, epochs=100, C=1.0)
    svm.fit(X_train_bow, y_train)
    y_pred = svm.predict(X_test_bow)
    metrics = compute_metrics(y_test, y_pred)
    print_results("SVM + Bag of Words", metrics)
    results.append(("SVM", "BoW", metrics))

    #  EXPERIMENT 2: TF-IDF
    print("\n[4] Feature Extraction: TF-IDF")
    tfidf = TfIdfVectorizer(max_features=3000, ngram_range=(1, 1))
    X_train_tfidf = tfidf.fit_transform(X_train)
    X_test_tfidf = tfidf.transform(X_test)

    # Naive Bayes + TF-IDF
    nb2 = NaiveBayesClassifier(alpha=1.0)
    nb2.fit(X_train_tfidf, y_train)
    y_pred = nb2.predict(X_test_tfidf)
    metrics = compute_metrics(y_test, y_pred)
    print_results("Naive Bayes + TF-IDF", metrics)
    results.append(("Naive Bayes", "TF-IDF", metrics))

    # Logistic Regression + TF-IDF
    lr2 = LogisticRegression(lr=1.0, epochs=100)
    lr2.fit(X_train_tfidf, y_train)
    y_pred = lr2.predict(X_test_tfidf)
    metrics = compute_metrics(y_test, y_pred)
    print_results("Logistic Regression + TF-IDF", metrics)
    results.append(("Logistic Regression", "TF-IDF", metrics))

    # SVM + TF-IDF
    svm2 = SVM(lr=0.01, epochs=100, C=1.0)
    svm2.fit(X_train_tfidf, y_train)
    y_pred = svm2.predict(X_test_tfidf)
    metrics = compute_metrics(y_test, y_pred)
    print_results("SVM + TF-IDF", metrics)
    results.append(("SVM", "TF-IDF", metrics))

    #  EXPERIMENT 3: Bigrams
    print("\n[5] Feature Extraction: Bigrams (TF-IDF)")
    tfidf_bi = TfIdfVectorizer(max_features=3000, ngram_range=(1, 2))
    X_train_bi = tfidf_bi.fit_transform(X_train)
    X_test_bi = tfidf_bi.transform(X_test)
    print(f"    Vocabulary size (with bigrams): {len(tfidf_bi.vocabulary)}")

    # Naive Bayes + Bigrams
    nb3 = NaiveBayesClassifier(alpha=1.0)
    nb3.fit(X_train_bi, y_train)
    y_pred = nb3.predict(X_test_bi)
    metrics = compute_metrics(y_test, y_pred)
    print_results("Naive Bayes + Bigrams TF-IDF", metrics)
    results.append(("Naive Bayes", "Bigrams", metrics))

    # Logistic Regression + Bigrams
    lr3 = LogisticRegression(lr=1.0, epochs=100)
    lr3.fit(X_train_bi, y_train)
    y_pred = lr3.predict(X_test_bi)
    metrics = compute_metrics(y_test, y_pred)
    print_results("Logistic Regression + Bigrams TF-IDF", metrics)
    results.append(("Logistic Regression", "Bigrams", metrics))

    # SVM + Bigrams
    svm3 = SVM(lr=0.01, epochs=100, C=1.0)
    svm3.fit(X_train_bi, y_train)
    y_pred = svm3.predict(X_test_bi)
    metrics = compute_metrics(y_test, y_pred)
    print_results("SVM + Bigrams TF-IDF", metrics)
    results.append(("SVM", "Bigrams", metrics))

    #  SUMMARY TABLE
    print("\n" + "=" * 70)
    print("   RESULTS SUMMARY")
    print("=" * 70)
    print(f"\n{'Classifier':<22} {'Features':<12} {'Accuracy':<10} {'Precision':<10} {'Recall':<10} {'F1':<10}")
    print("-" * 70)

    for clf, feat, m in results:
        print(f"{clf:<22} {feat:<12} {m['accuracy']:<10.4f} {m['precision']:<10.4f} {m['recall']:<10.4f} {m['f1']:<10.4f}")

    # Find best
    best = max(results, key=lambda x: x[2]['f1'])
    print(f"\nBest Model: {best[0]} + {best[1]} (F1: {best[2]['f1']:.4f})")

    # Return best model for interactive use
    return tfidf, lr2  # Return TF-IDF vectorizer and best model


def interactive_mode(vectorizer, model):
    """Interactive prediction mode."""
    print("\n" + "=" * 70)
    print("   INTERACTIVE CLASSIFICATION")
    print("   Enter text to classify as Sports or Politics")
    print("   Type 'quit' to exit")
    print("=" * 70)

    labels = {0: "POLITICS", 1: "SPORTS"}

    while True:
        try:
            text = input("\nEnter text: ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\nGoodbye!")
            break

        if not text:
            continue

        if text.lower() in ['quit', 'exit', 'q']:
            print("Goodbye!")
            break

        # Transform and predict
        X = vectorizer.transform([text])
        prediction = model.predict(X)[0]

        print(f"Prediction: {labels[prediction]}")


def main():
    vectorizer, model = run_experiment()
    interactive_mode(vectorizer, model)


if __name__ == "__main__":
    main()

   SPORTS vs POLITICS CLASSIFIER
   Comparing: Naive Bayes, Logistic Regression, SVM
   Features: Bag of Words, TF-IDF, N-grams

[1] Loading AG News dataset...
    Training samples: 1600 (Politics: 800, Sports: 800)
    Test samples: 400 (Politics: 200, Sports: 200)

[2] Feature Extraction: Bag of Words (Unigrams)
    Vocabulary size: 3000

[3] Training classifiers on BoW features...

Naive Bayes + Bag of Words
----------------------------------------
  Accuracy:  0.9400
  Precision: 0.9112
  Recall:    0.9750
  F1-Score:  0.9420
  Confusion Matrix:
    TP=195, TN=181, FP=19, FN=5

Logistic Regression + Bag of Words
----------------------------------------
  Accuracy:  0.9100
  Precision: 0.8981
  Recall:    0.9250
  F1-Score:  0.9113
  Confusion Matrix:
    TP=185, TN=179, FP=21, FN=15

SVM + Bag of Words
----------------------------------------
  Accuracy:  0.9275
  Precision: 0.9091
  Recall:    0.9500
  F1-Score:  0.9291
  Confusion Matrix:
    TP=190, TN=181, FP=19, FN=10

[4] Fea